# 01 — From room-temperature resistance to qubit frequency

A Josephson junction's resistance, measured at room temperature with a probe needle, already tells you
most of what you need to predict the qubit's frequency after cooldown. The chain is:

1. **Gap**: Δ = 1.764 · k_B · T_c (BCS).
2. **Critical current** (Ambegaokar–Baratoff): I_c = π Δ / (2 e R_n). Higher resistance → weaker junction.
3. **Josephson energy**: E_J = I_c Φ₀ / 2π.
4. **Charging energy** of the qubit capacitor: E_C = e² / (2 C_q).
5. **Transmon frequency**: f₀₁ ≈ (√(8 E_J E_C) − E_C) / h.

Put together this is Eq. 1 of Mayer, Weber et al., [arXiv:2505.08424](https://arxiv.org/abs/2505.08424) (Fraunhofer EMFT, 2025),
which uses a fitted effective T_c = 0.71 K ([Lang et al., arXiv:2504.18173](https://arxiv.org/abs/2504.18173), Sec. IV.2)
and design capacitance C_q = 86 fF.

In [1]:
from yieldkit.physics import (gap_from_tc, f01_from_rn, rn_target_for_f01,
                              charging_energy, H_PLANCK, E_CHARGE)

TC_FIT = 0.71      # K, fitted effective Tc (Lang et al., arXiv:2504.18173, Sec. IV.2)
CQ = 86e-15        # F, design capacitance (arXiv:2505.08424, Sec. III.1)
gap_fit = gap_from_tc(TC_FIT)

print(f"Effective gap          : {gap_fit / E_CHARGE * 1e6:6.1f} µeV")
print(f"E_C / h (our calc)     : {charging_energy(CQ) / H_PLANCK / 1e6:6.1f} MHz")
print(f"f01 at Rn = 7.1 kΩ     : {f01_from_rn(7.1e3, CQ, gap_fit) / 1e9:6.3f} GHz  (paper design: 4.42 GHz)")
print(f"Rn needed for 4.42 GHz : {rn_target_for_f01(4.42e9, CQ, gap_fit) / 1e3:6.2f} kΩ  (paper: ≈ 7.1 kΩ)")

Effective gap          :  107.9 µeV
E_C / h (our calc)     :  225.2 MHz
f01 at Rn = 7.1 kΩ     :  4.397 GHz  (paper design: 4.42 GHz)
Rn needed for 4.42 GHz :   7.03 kΩ  (paper: ≈ 7.1 kΩ)


## Why calibration per fab matters

What if we had used the textbook gap from bulk aluminium (T_c ≈ 1.2 K, the literature value quoted by Lang et al.)
instead of Fraunhofer's fitted 0.71 K? Same resistance, same capacitor:

In [2]:
TC_BULK_AL = 1.2   # K, literature value for Al quoted in Lang et al., arXiv:2504.18173, Sec. IV.2
gap_bulk = gap_from_tc(TC_BULK_AL)

rn = 7.1e3
f_fit = f01_from_rn(rn, CQ, gap_fit)
f_bulk = f01_from_rn(rn, CQ, gap_bulk)
print(f"Bulk-Al gap: {gap_bulk / E_CHARGE * 1e6:.1f} µeV")
print(f"f01 with fitted Tc : {f_fit / 1e9:.3f} GHz")
print(f"f01 with bulk Tc   : {f_bulk / 1e9:.3f} GHz")
print(f"Textbook gap overestimates f01 by {100 * (f_bulk / f_fit - 1):.1f} %")

Bulk-Al gap: 182.4 µeV
f01 with fitted Tc : 4.397 GHz
f01 with bulk Tc   : 5.785 GHz
Textbook gap overestimates f01 by 31.5 %


**Lesson:** using the textbook gap gives a large error, while the paper's prediction accuracy is 1.6%.
The effective T_c has to be calibrated for each fab and process. Lang et al. report that this offset is stable
between processing runs, which is what makes a single calibration useful.

## Teaching cell: where does √(8 E_J E_C) − E_C come from?

The transmon Hamiltonian in the charge basis |n⟩ (n = number of Cooper pairs that have tunnelled) is

H = 4 E_C (n − n_g)² − (E_J / 2) Σ ( |n⟩⟨n+1| + |n+1⟩⟨n| )

It is a tridiagonal matrix. Diagonalise it and take the gap between the two lowest levels.
When E_J ≫ E_C the potential is nearly a harmonic well (frequency √(8 E_J E_C)), and the cosine's
anharmonicity pulls the 0→1 level down by about E_C. That gives the formula.

In [3]:
import numpy as np

def transmon_levels(ej, ec, ng=0.25, ncut=30):
    """Eigen-energies of the charge-basis transmon Hamiltonian (same units as ej, ec)."""
    n = np.arange(-ncut, ncut + 1)
    H = np.diag(4 * ec * (n - ng) ** 2)
    off = -ej / 2 * np.ones(len(n) - 1)
    H += np.diag(off, 1) + np.diag(off, -1)
    return np.linalg.eigvalsh(H)

ec_ghz = charging_energy(CQ) / H_PLANCK / 1e9
print(" EJ/EC   exact f01   √(8EJEC)−EC   difference")
for ratio in (20, 50, 100):
    ej_ghz = ratio * ec_ghz
    E = transmon_levels(ej_ghz, ec_ghz)
    exact = E[1] - E[0]
    asym = np.sqrt(8 * ej_ghz * ec_ghz) - ec_ghz
    print(f"{ratio:5d}   {exact:7.4f} GHz   {asym:7.4f} GHz   {1e3 * (asym - exact):5.1f} MHz")

 EJ/EC   exact f01   √(8EJEC)−EC   difference
   20    2.6006 GHz    2.6238 GHz    23.2 MHz
   50    4.2664 GHz    4.2795 GHz    13.1 MHz
  100    6.1366 GHz    6.1454 GHz     8.8 MHz


The simple formula overestimates f₀₁ by roughly 10–25 MHz in this range (about 0.3% at Fraunhofer's E_J/E_C ≈ 53).
That is well below the 1.6% prediction accuracy reported in the paper, so the closed form is good enough for wafer-level prediction.